# **Analítica Computacional para la Toma de Decisiones**

## Taller 10: Visualización

Daniel Benavides -202220428

<d.benavidess@uniandes.edu.co>

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import altair as alt

from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

alt.data_transformers.enable('vegafusion')

import warnings
warnings.filterwarnings('ignore')  

In [ ]:
# Read data

df = pd.read_csv("people.csv")
df.head()

In [ ]:
# Categorical and numerical variables
df.info()

In [ ]:
categorical_vars = df.select_dtypes(include=['object']).columns
numerical_vars = df.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
categorical_vars

In [ ]:
numerical_vars

In [ ]:
variables = ["OverTime", "WorkLifeBalance", "YearsSinceLastPromotion"]

In [ ]:
# Altair pairplot
num_vars = [
    'Age',
    'MonthlyIncome',
    'TotalWorkingYears',
    'YearsAtCompany',
    'YearsSinceLastPromotion',
    'WorkLifeBalance',
    'JobSatisfaction'
]

color_scale = alt.Scale(
    domain=['No', 'Yes'],
    range=['steelblue', 'crimson']
)

pairplot = (
    alt.Chart(df)
    .mark_circle(opacity=0.4, size=18)
    .encode(
        x=alt.X(alt.repeat('column'), type='quantitative'),
        y=alt.Y(alt.repeat('row'),    type='quantitative'),
        color=alt.Color('Attrition:N', scale=color_scale,
                        legend=alt.Legend(title='Attrition')),
        tooltip=['Age', 'MonthlyIncome', 'Attrition']
    )
    .properties(width=120, height=120)
    .repeat(
        row=num_vars,
        column=num_vars
    )
)

pairplot

In [ ]:
base = (
    alt.Chart(df)
    .transform_aggregate(count_="count()", groupby=["Department", "Attrition"])
    .transform_stack(
        stack="count_",
        as_=["stack_count_Department1", "stack_count_Department2"],
        offset="normalize",
        sort=[alt.SortField("Department", "ascending")],
        groupby=[],
    )
    .transform_window(
        x="min(stack_count_Department1)",
        x2="max(stack_count_Department2)",
        rank_Attrition="dense_rank()",
        distinct_Attrition="distinct(Attrition)",
        groupby=["Department"],
        frame=[None, None],
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_window(
        rank_Department="dense_rank()",
        frame=[None, None],
        sort=[alt.SortField("Department", "ascending")],
    )
    .transform_stack(
        stack="count_",
        groupby=["Department"],
        as_=["y", "y2"],
        offset="normalize",
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_calculate(
        ny="datum.y + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        ny2="datum.y2 + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        nx="datum.x + (datum.rank_Department - 1) * 0.01",
        nx2="datum.x2 + (datum.rank_Department - 1) * 0.01",
        xc="(datum.nx + datum.nx2) / 2",
        yc="(datum.ny + datum.ny2) / 2",
    )
)

rect = base.mark_rect().encode(
    x=alt.X("nx:Q").axis(None),
    x2="nx2",
    y=alt.Y("ny:Q").axis(None),
    y2="ny2",
    color=alt.Color("Department:N", legend=None),         
    opacity=alt.condition(
        alt.datum.Attrition == "Yes",
        alt.value(0.8), 
        alt.value(0.6),  
    ),
    tooltip=[
        alt.Tooltip("Department:N",  title="Departamento"),
        alt.Tooltip("Attrition:N",   title="Attrition"),
        alt.Tooltip("count_:Q",      title="Empleados"),
    ],
)

# Etiqueta: "Yes" / "No"
text_label = base.mark_text(
    baseline="middle", align="center",
    fontSize=12, fontWeight="bold", dy=-10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="Attrition:N",
    color=alt.value("white"),
)

# Número de empleados en cada bloque
text_count = base.mark_text(
    baseline="middle", align="center",
    fontSize=14, fontWeight="bold", dy=10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="count_:Q",
    color=alt.value("white"),
)

mosaic = (rect + text_label + text_count).properties(width=620, height=420)

dept_labels = (
    base.mark_text(baseline="middle", align="center", fontSize=15, fontWeight="bold")
    .encode(
        x=alt.X("min(xc):Q").axis(None),
        color=alt.Color("Department:N", legend=None),
        text="Department:N",
    )
    .properties(width=620)
)

(
    (dept_labels & mosaic)
    .resolve_scale(x="shared")
    .configure_view(stroke="")
    .configure_concat(spacing=2)
    .configure_axis(domain=False, ticks=False, labels=False, grid=False)
)

In [ ]:
# Porcentajes de empleados que dejaron la empresa por departamento
attrition_percent = (
    df.groupby("Department")["Attrition"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="AttritionPercent")
)

attrition_percent.round(2)

In [ ]:
base_bt = (
    alt.Chart(df)
    .transform_calculate(
        BusinessTravelES="""
            datum.BusinessTravel === 'Travel_Rarely' ? 'Viaja Raramente' :
            datum.BusinessTravel === 'Travel_Frequently' ? 'Viaja Frecuentemente' :
            'No Viaja'
        """
    )
    .transform_aggregate(count_="count()", groupby=["BusinessTravelES", "Attrition"])
    .transform_stack(
        stack="count_",
        as_=["stack_count_BT1", "stack_count_BT2"],
        offset="normalize",
        sort=[alt.SortField("BusinessTravelES", "ascending")],
        groupby=[],
    )
    .transform_window(
        x="min(stack_count_BT1)",
        x2="max(stack_count_BT2)",
        rank_Attrition="dense_rank()",
        distinct_Attrition="distinct(Attrition)",
        groupby=["BusinessTravelES"],
        frame=[None, None],
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_window(
        rank_BusinessTravelES="dense_rank()",
        frame=[None, None],
        sort=[alt.SortField("BusinessTravelES", "ascending")],
    )
    .transform_stack(
        stack="count_",
        groupby=["BusinessTravelES"],
        as_=["y", "y2"],
        offset="normalize",
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_calculate(
        ny="datum.y + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        ny2="datum.y2 + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        nx="datum.x + (datum.rank_BusinessTravelES - 1) * 0.01",
        nx2="datum.x2 + (datum.rank_BusinessTravelES - 1) * 0.01",
        xc="(datum.nx + datum.nx2) / 2",
        yc="(datum.ny + datum.ny2) / 2",
    )
)

rect_bt = base_bt.mark_rect().encode(
    x=alt.X("nx:Q").axis(None),
    x2="nx2",
    y=alt.Y("ny:Q").axis(None),
    y2="ny2",
    color=alt.Color("BusinessTravelES:N", legend=None),
    opacity=alt.condition(
        alt.datum.Attrition == "Yes",
        alt.value(0.8),
        alt.value(0.6),
    ),
    tooltip=[
        alt.Tooltip("BusinessTravelES:N", title="Frecuencia de Viaje"),
        alt.Tooltip("Attrition:N",        title="Attrition"),
        alt.Tooltip("count_:Q",           title="Empleados"),
    ],
)

text_label_bt = base_bt.mark_text(
    baseline="middle", align="center",
    fontSize=12, fontWeight="bold", dy=-10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="Attrition:N",
    color=alt.value("white"),
)

text_count_bt = base_bt.mark_text(
    baseline="middle", align="center",
    fontSize=14, fontWeight="bold", dy=10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="count_:Q",
    color=alt.value("white"),
)

mosaic_bt = (rect_bt + text_label_bt + text_count_bt).properties(width=620, height=420)

labels_bt = (
    base_bt.mark_text(baseline="middle", align="center", fontSize=15, fontWeight="bold")
    .encode(
        x=alt.X("min(xc):Q").axis(None),
        color=alt.Color("BusinessTravelES:N", legend=None),
        text="BusinessTravelES:N",
    )
    .properties(width=620)
)

mosaic_businesstravel = (
    (labels_bt & mosaic_bt)
    .resolve_scale(x="shared")
    .configure_view(stroke="")
    .configure_concat(spacing=2)
    .configure_axis(domain=False, ticks=False, labels=False, grid=False)
)

mosaic_businesstravel

In [ ]:
# Porcentajes de empleados que dejaron la empresa por tipo de viaje
attrition_percent_bt = (
    df.groupby("BusinessTravel")["Attrition"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="AttritionPercent")
)

attrition_percent_bt.round(2)

In [ ]:
base_ms = (
    alt.Chart(df)
    .transform_aggregate(count_="count()", groupby=["MaritalStatus", "Attrition"])
    .transform_stack(
        stack="count_",
        as_=["stack_count_MS1", "stack_count_MS2"],
        offset="normalize",
        sort=[alt.SortField("MaritalStatus", "ascending")],
        groupby=[],
    )
    .transform_window(
        x="min(stack_count_MS1)",
        x2="max(stack_count_MS2)",
        rank_Attrition="dense_rank()",
        distinct_Attrition="distinct(Attrition)",
        groupby=["MaritalStatus"],
        frame=[None, None],
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_window(
        rank_MaritalStatus="dense_rank()",
        frame=[None, None],
        sort=[alt.SortField("MaritalStatus", "ascending")],
    )
    .transform_stack(
        stack="count_",
        groupby=["MaritalStatus"],
        as_=["y", "y2"],
        offset="normalize",
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_calculate(
        ny="datum.y + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        ny2="datum.y2 + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        nx="datum.x + (datum.rank_MaritalStatus - 1) * 0.01",
        nx2="datum.x2 + (datum.rank_MaritalStatus - 1) * 0.01",
        xc="(datum.nx + datum.nx2) / 2",
        yc="(datum.ny + datum.ny2) / 2",
    )
)

rect_ms = base_ms.mark_rect().encode(
    x=alt.X("nx:Q").axis(None),
    x2="nx2",
    y=alt.Y("ny:Q").axis(None),
    y2="ny2",
    color=alt.Color("MaritalStatus:N", legend=None),
    opacity=alt.condition(
        alt.datum.Attrition == "Yes",
        alt.value(0.8),
        alt.value(0.6),
    ),
    tooltip=[
        alt.Tooltip("MaritalStatus:N", title="Estado Civil"),
        alt.Tooltip("Attrition:N",     title="Attrition"),
        alt.Tooltip("count_:Q",        title="Empleados"),
    ],
)

text_label_ms = base_ms.mark_text(
    baseline="middle", align="center",
    fontSize=12, fontWeight="bold", dy=-10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="Attrition:N",
    color=alt.value("white"),
)

text_count_ms = base_ms.mark_text(
    baseline="middle", align="center",
    fontSize=14, fontWeight="bold", dy=10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="count_:Q",
    color=alt.value("white"),
)

mosaic_ms = (rect_ms + text_label_ms + text_count_ms).properties(width=620, height=420)

labels_ms = (
    base_ms.mark_text(baseline="middle", align="center", fontSize=15, fontWeight="bold")
    .encode(
        x=alt.X("min(xc):Q").axis(None),
        color=alt.Color("MaritalStatus:N", legend=None),
        text="MaritalStatus:N",
    )
    .properties(width=620)
)

mosaic_maritalstatus = (
    (labels_ms & mosaic_ms)
    .resolve_scale(x="shared")
    .configure_view(stroke="")
    .configure_concat(spacing=2)
    .configure_axis(domain=False, ticks=False, labels=False, grid=False)
)

mosaic_maritalstatus

In [ ]:
# Porcentajes de empleados que dejaron la empresa por estado civil
attrition_percent_ms = (
    df.groupby("MaritalStatus")["Attrition"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="AttritionPercent")
)

attrition_percent_ms.round(2)